In [30]:
import json
import os
class Transaction:
    def __init__(self, date, amount, category, payment_type):
        self._date = date
        self._amount = amount
        self._category = category
        self._payment_type = payment_type

    # --- date ---
    @property
    def date(self):
        return self._date

    @date.setter
    def date(self, value):
        self._date = value

    # --- amount ---
    @property
    def amount(self):
        return self._amount

    @amount.setter
    def amount(self, value):
        if value < 0:
            raise ValueError("Amount cannot be negative")
        self._amount = value

    # --- category ---
    @property
    def category(self):
        return self._category

    @category.setter
    def category(self, value):
        self._category = value

    # --- payment_type ---
    @property
    def payment_type(self):
        return self._payment_type

    @payment_type.setter
    def payment_type(self, value):
        self._payment_type = value

    def to_dict(self):
        return {
            "date": self._date,
            "amount": self._amount,
            "category": self._category,
            "payment_type": self._payment_type
        }

    @staticmethod
    def from_dict(data):
        return Transaction(
            data["date"],
            data["amount"],
            data["category"],
            data["payment_type"]
        )

    def __str__(self):
        return f"{self._date} | {self._category:<12} | {self._payment_type:<10} | {self._amount}"
    def __repr__(self):
        return self.__str__()

In [3]:
class Category:
    categories = [
        "Food",
        "Transport",
        "Bills",
        "Salary",
        "Shopping",
        "Entertainment"
    ]

In [4]:
class StorageManager:
    def __init__(self, filepath="transactions.json"):
        self.filepath = filepath
    def save(self, transactions):
        data = []
        for t in transactions:
            data.append(t.to_dict())

        file = open(self.filepath, mode='w')
        json.dump(data, file, indent=4)
        file.close()

    def load(self):
        if os.path.exists(self.filepath) == False:
            return []

        file = open(self.filepath, mode='r')
        data = json.load(file)
        file.close()

        transactions = []
        for item in data:
            t = Transaction_from_dict(item)
            transactions.append(t)
        return transactions

In [15]:
sm = StorageManager("test.json")

t1 = Transaction("2025-08-31", 500, "Food", "Cash")
t2 = Transaction("2025-08-30", 2000, "Rent", "Bank Transfer")

sm.save([t1, t2])
print("Saved!")

loaded = sm.load()
for t in loaded:
    print(t)

Saved!
2025-08-31 | Food         | Cash       | 500
2025-08-30 | Rent         | Bank Transfer | 2000


In [51]:
class Budget:
    def __init__(self):
        self._limits = {}
        self._savings_goal = None

    @property
    def limits(self):
        return self._limits

    def set_limit(self, category, amount):
        if amount < 0:
            raise ValueError("Limit cannot be negative")
        self._limits[category] = amount

    def get_limit(self, category):
        if category in self._limits:
            return self._limits[category]
        else:
            return None

    def set_savings_goal(self, amount):
        if amount < 0:
            raise ValueError("Savings goal cannot be negative")
        self._savings_goal = amount

    def check_savings_progress(self, current_balance):
        if self._savings_goal is None:
            return "No savings goal set."

        if self._savings_goal == 0:
            percent = 0
        else:
            percent = (current_balance / self._savings_goal) * 100

        if percent > 100:
            percent = 100
        if percent < 0:
            percent = 0

        bar_filled = int(percent // 10)
        bar_empty = 10 - bar_filled
        bar = "#" * bar_filled + "-" * bar_empty

        return f"Savings Goal: {current_balance}/{self._savings_goal} [{bar}] {percent:.0f}%"

    def check_alert(self, category, spent):
        limit = self.get_limit(category)
        if limit is None or limit == 0:
            return None

        percent = (spent / limit) * 100

        if percent >= 100:
            return f"[ALERT] {category} is at {percent:.0f}% of budget (OVER LIMIT)"
        elif percent >= 80:
            return f"[WARNING] {category} is at {percent:.0f}% of budget"
        else:
            return None

    def monthly_summary(self, transactions, month, year):
        totals = {}

        for t in transactions:
            parts = t.date.split("-")
            t_year = int(parts[0])
            t_month = int(parts[1])

            if t_year == year and t_month == month:
                if t.category in totals:
                    totals[t.category] = totals[t.category] + t.amount
                else:
                    totals[t.category] = t.amount

        print(f"{'Category':<15}{'Spent':<10}{'Limit':<10}{'Status':<10}")
        print("-" * 45)

        for category in totals:
            spent = totals[category]
            limit = self.get_limit(category)

            if limit is None:
                limit_display = "N/A"
                status = "-"
            else:
                limit_display = str(limit)
                if spent > limit:
                    status = "OVER"
                else:
                    status = "OK"

            print(f"{category:<15}{spent:<10}{limit_display:<10}{status:<10}")

            alert = self.check_alert(category, spent)
            if alert:
                print("   " + alert)

        return totals

In [7]:
t1 = Transaction("2025-08-05", 500, "Food", "Cash")
t2 = Transaction("2025-08-10", 700, "Food", "Card")
t3 = Transaction("2025-08-01", 5000, "Rent", "Bank Transfer")
t4 = Transaction("2025-08-15", 300, "Entertainment", "Cash")
t5 = Transaction("2025-07-01", 200, "Food", "Cash")
transactions = [t1, t2, t3, t4, t5]

sm = StorageManager("test.json")
sm.save(transactions)
print("Saved to test.json!")

Saved to test.json!


In [14]:
loaded = sm.load()
for t in loaded:
    print(t)

[INCOME]  2025-08-01 | Salary       | Bank Transfer | +50000
[EXPENSE] 2025-08-05 | Food         | Cash         | -1200


In [9]:
#BONUS CONTENT STARTS FROM HERE
class Income(Transaction):
    def __init__(self, date, amount, category, payment_type):
        super().__init__(date, amount, category, payment_type)

    def apply(self, balance):
        return balance + self.amount   

    def to_dict(self):
        data = super().to_dict()       
        data["type"] = "Income"        
        return data

    def __str__(self):
        return f"[INCOME]  {self.date} | {self.category:<12} | {self.payment_type:<12} | +{self.amount}"


class Expense(Transaction):
    def __init__(self, date, amount, category, payment_type):
        super().__init__(date, amount, category, payment_type)

    def apply(self, balance):
        return balance - self.amount  

    def to_dict(self):
        data = super().to_dict()
        data["type"] = "Expense"
        return data

    def __str__(self):
        return f"[EXPENSE] {self.date} | {self.category:<12} | {self.payment_type:<12} | -{self.amount}"

In [40]:
i = Income("2025-08-01", 50000, "Salary", "Bank Transfer")
e = Expense("2025-08-05", 1200, "Food", "Cash")

print(i)
print(e)

balance = 0
balance = i.apply(balance)
balance = e.apply(balance)
print("Balance:", balance)

print(i.to_dict())
sm = JSONStorage("test.json")
sm.save(transactions)
print("Saved to test.json!")

[INCOME]  2025-08-01 | Salary       | Bank Transfer | +50000
[EXPENSE] 2025-08-05 | Food         | Cash         | -1200
Balance: 48800
{'date': '2025-08-01', 'amount': 50000, 'category': 'Salary', 'payment_type': 'Bank Transfer', 'type': 'Income'}
Saved to test.json!


In [11]:
def Transaction_from_dict(data):
    if data.get("type") == "Income":
        return Income(data["date"], data["amount"], data["category"], data["payment_type"])
    elif data.get("type") == "Expense":
        return Expense(data["date"], data["amount"], data["category"], data["payment_type"])
    else:
        return Transaction(data["date"], data["amount"], data["category"], data["payment_type"])

In [35]:
sm = JSONStorage("test.json") 

i = Income("2025-08-02", 50000, "Salary", "Bank Transfer")
e = Expense("2025-08-05", 1200, "Food", "Cash")
sm.save([i, e])

loaded = sm.load()
for t in loaded:
    print(t)
    print(type(t))

[INCOME]  2025-08-02 | Salary       | Bank Transfer | +50000
<class '__main__.Income'>
[EXPENSE] 2025-08-05 | Food         | Cash         | -1200
<class '__main__.Expense'>


In [23]:
from abc import ABC, abstractmethod
import csv

class StorageManager(ABC):
    def __init__(self, filepath):
        self.filepath = filepath

    @abstractmethod
    def save(self, transactions):
        pass

    @abstractmethod
    def load(self):
        pass


class JSONStorage(StorageManager):
    def __init__(self, filepath="transactions.json"):
        super().__init__(filepath)

    def save(self, transactions):
        data = []
        for t in transactions:
            data.append(t.to_dict())

        file = open(self.filepath, mode='w')
        json.dump(data, file, indent=4)
        file.close()

    def load(self):
        if os.path.exists(self.filepath) == False:
            return []

        file = open(self.filepath, mode='r')
        data = json.load(file)
        file.close()

        transactions = []
        for item in data:
            t = Transaction_from_dict(item)
            transactions.append(t)
        return transactions


class CSVStorage(StorageManager):
    def __init__(self, filepath="transactions.csv"):
        super().__init__(filepath)

    def save(self, transactions):
        file = open(self.filepath, mode='w', newline='')
        writer = csv.writer(file)
        writer.writerow(["type", "date", "amount", "category", "payment_type"])   

        for t in transactions:
            d = t.to_dict()
            writer.writerow([d.get("type", "Transaction"), d["date"], d["amount"], d["category"], d["payment_type"]])

        file.close()

    def load(self):
        if os.path.exists(self.filepath) == False:
            return []

        file = open(self.filepath, mode='r', newline='')
        reader = csv.DictReader(file)

        transactions = []
        for row in reader:
            row["amount"] = float(row["amount"])  
            t =Transaction_from_dict(row)
            transactions.append(t)

        file.close()
        return transactions

In [38]:
json_storage = JSONStorage("transactions.json")
csv_storage = CSVStorage("transactions.csv")

transactions = json_storage.load()
csv_storage.save(transactions)

print("Converted! Check transactions.csv")

Converted! Check transactions.csv


In [56]:
def calculate_balance(transactions):
    balance = 0
    for t in transactions:
        balance = t.apply(balance)
    return balance


def main():
    sm = JSONStorage("transactions.json") 
    transactions = sm.load()
    b = Budget()
    b.set_limit("Food", 5000)      
    b.set_limit("Rent", 15000)
    b.set_savings_goal(20000)

    while True:
        print()
        print("1. Add transaction")
        print("2. List transactions")
        print("3. Show summary")
        print("4. Show balance")
        print("5. Show savings goal progress")
        print("6. Exit")
        choice = input("Enter your choice: ")

        if choice == "1":
            t_type = input("Type (income/expense): ").strip().lower()
            if t_type not in ["income", "expense"]:
                print("Invalid type. Please enter 'income' or 'expense'.")
                continue

            date = input("Date (YYYY-MM-DD): ")
            parts = date.split("-")
            if len(parts) != 3:
                print("Invalid date format. Please use YYYY-MM-DD.")
                continue

            try:
                int(parts[0])
                int(parts[1])
                int(parts[2])
            except ValueError:
                print("Invalid date format. Please use YYYY-MM-DD.")
                continue

            try:
                amount = float(input("Amount: "))
            except ValueError:
                print("Invalid amount. Please enter a number.")
                continue

            print("Available categories:", Category.categories)
            category = input("Category: ")
            if category not in Category.categories:
                print("Invalid category. Please choose from the list above.")
                continue

            payment_type = input("Payment type: ")

            try:
                if t_type == "income":
                    new_t = Income(date, amount, category, payment_type)
                else:
                    new_t = Expense(date, amount, category, payment_type)
            except ValueError as e:
                print("Error:", e)
                continue

            transactions.append(new_t)
            sm.save(transactions)
            print("Transaction added and saved!")

            t_date_parts = date.split("-")
            check_year = int(t_date_parts[0])
            check_month = int(t_date_parts[1])

            month_total = 0
            for tx in transactions:
                tx_parts = tx.date.split("-")
                if int(tx_parts[0]) == check_year and int(tx_parts[1]) == check_month and tx.category == category:
                    month_total = month_total + tx.amount

            alert = b.check_alert(category, month_total)
            if alert:
                print(alert)

        elif choice == "2":
            if len(transactions) == 0:
                print("No transactions yet.")
            else:
                for t in transactions:
                    print(t)

        elif choice == "3":
            try:
                month = int(input("Month (1-12): "))
                year = int(input("Year (e.g. 2025): "))
            except ValueError:
                print("Invalid month/year.")
                continue

            b.monthly_summary(transactions, month, year)

        elif choice == "4":
            balance = calculate_balance(transactions)
            print(f"Current balance: {balance}")

        elif choice == "5":
            balance = calculate_balance(transactions)
            print(b.check_savings_progress(balance))

        elif choice == "6":
            sm.save(transactions)
            print("Goodbye!")
            break

        else:
            print("Invalid choice, try again.")


main()
sm = JSONStorage("transactions.json")   


1. Add transaction
2. List transactions
3. Show summary
4. Show balance
5. Show savings goal progress
6. Exit


Enter your choice:  5


Savings Goal: 93550.0/20000 [##########] 100%

1. Add transaction
2. List transactions
3. Show summary
4. Show balance
5. Show savings goal progress
6. Exit


Enter your choice:  5


Savings Goal: 93550.0/20000 [##########] 100%

1. Add transaction
2. List transactions
3. Show summary
4. Show balance
5. Show savings goal progress
6. Exit


Enter your choice:  6


Goodbye!
